# 🏥 Health Insurance Claims Cost Predictor
**Author:** Hillary Onah — Finance & Data Science Analyst, DHIN  
**Date:** June 2026  

### Objective
Identify the key drivers of health insurance claim costs and build a machine learning model that predicts a patient's expected claim amount from their demographic and lifestyle profile.

### Models used
- Linear Regression — simple, interpretable baseline
- Random Forest — powerful ensemble model

### Dataset
US Medical Insurance Costs — 1,338 records, 7 features (age, sex, BMI, children, smoker status, region, charges)

---
## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

# Render charts inline inside the notebook
%matplotlib inline
plt.rcParams['figure.dpi'] = 130
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')

print('✅ Libraries loaded')

---
## 2. Load and inspect the data

In [ ]:
df = pd.read_csv('medical_cost.csv')
df = df.drop_duplicates().dropna()

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}\n')
df.head()

In [ ]:
# Summary statistics — min, max, mean, std for every column
df.describe().round(2)

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Chart 1: Distribution of claim costs
fig, ax = plt.subplots()
sns.histplot(df['charges'], bins=50, kde=True, color='steelblue', ax=ax)
ax.set_title('Distribution of insurance claim costs', fontsize=13, fontweight='bold')
ax.set_xlabel('Claim cost (USD)')
ax.set_ylabel('Number of claims')
plt.tight_layout()
plt.savefig('chart1_cost_distribution.png', dpi=150)
plt.show()

print(f"Mean:   ${df['charges'].mean():,.0f}")
print(f"Median: ${df['charges'].median():,.0f}")
print(f"Note: mean > median confirms right skew — outliers pulling average up")

In [ ]:
# Chart 2: Average cost by age group
df['age_group'] = pd.cut(df['age'],
    bins=[0, 25, 35, 45, 55, 100],
    labels=['Under 25', '25-34', '35-44', '45-54', '55+'])

age_cost = df.groupby('age_group', observed=True)['charges'].mean().reset_index()

fig, ax = plt.subplots()
sns.barplot(data=age_cost, x='age_group', y='charges', palette='Blues_d', ax=ax)
ax.set_title('Average claim cost by age group', fontsize=13, fontweight='bold')
ax.set_xlabel('Age group')
ax.set_ylabel('Average claim (USD)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('chart2_age_group_costs.png', dpi=150)
plt.show()

In [ ]:
# Chart 3: Smoker vs non-smoker cost comparison
smoker_cost = df.groupby('smoker')['charges'].mean()
ratio = smoker_cost['yes'] / smoker_cost['no']

fig, ax = plt.subplots(figsize=(6, 4))
colors = ['#2ECC71', '#E74C3C']
smoker_cost.plot(kind='bar', ax=ax, color=colors, width=0.5)
ax.set_title('Average claim cost: smoker vs non-smoker', fontsize=13, fontweight='bold')
ax.set_ylabel('Average claim (USD)')
ax.set_xticklabels(['Non-smoker', 'Smoker'], rotation=0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('chart3_smoker_cost.png', dpi=150)
plt.show()

print(f'Non-smoker avg: ${smoker_cost["no"]:,.0f}')
print(f'Smoker avg:     ${smoker_cost["yes"]:,.0f}')
print(f'Smokers cost {ratio:.1f}x more than non-smokers')

In [ ]:
# Chart 4: Scatter — age vs cost, coloured by smoker status
# This reveals the two-band structure: smokers form an elevated cluster at every age
fig, ax = plt.subplots()
for status, color, label in [('yes', '#E74C3C', 'Smoker'), ('no', '#4472C4', 'Non-smoker')]:
    subset = df[df['smoker'] == status]
    ax.scatter(subset['age'], subset['charges'], alpha=0.4, s=18,
               color=color, label=label)
ax.set_title('Age vs claim cost by smoker status', fontsize=13, fontweight='bold')
ax.set_xlabel('Age')
ax.set_ylabel('Claim cost (USD)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

print('Key insight: Two distinct bands visible.')
print('Smoking adds a roughly fixed cost premium across all ages.')

---
## 4. Prepare data for machine learning

In [ ]:
# Encode categorical variables as numbers
# ML models need numbers — text categories must be converted

df_ml = df[['age','sex','bmi','children','smoker','region','charges']].copy()

le = LabelEncoder()
df_ml['sex']    = le.fit_transform(df_ml['sex'])     # female=0, male=1
df_ml['smoker'] = le.fit_transform(df_ml['smoker'])  # no=0, yes=1
df_ml['region'] = le.fit_transform(df_ml['region'])  # numeric encoding

print('Encoding complete:')
print('  sex    → female=0, male=1')
print('  smoker → no=0, yes=1')
print('  region → numeric (0-3)\n')
df_ml.head()

In [ ]:
# Define features and target
X = df_ml[['age', 'sex', 'bmi', 'children', 'smoker', 'region']]
y = df_ml['charges']

# Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples: {len(X_train)}')
print(f'Test samples:     {len(X_test)}')
print(f'\nFeatures: {X.columns.tolist()}')
print(f'Target:   charges (claim cost USD)')

---
## 5. Model A — Linear Regression

**What it does:** Finds the best straight-line relationship between each feature and claim cost.  
**Strength:** Simple, fast, highly interpretable — you can read exactly how much each variable contributes.  
**Limitation:** Assumes relationships are linear, which isn't always true in health data.

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr   = r2_score(y_test, y_pred_lr)

print('LINEAR REGRESSION RESULTS')
print('-' * 35)
print(f'R²   (% variation explained): {r2_lr*100:.1f}%')
print(f'MAE  (average $ error):       ${mae_lr:,.0f}')
print(f'RMSE (error penalising outliers): ${rmse_lr:,.0f}')

In [ ]:
# Show the model coefficients — what each variable contributes to cost
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr.coef_
}).sort_values('Coefficient', ascending=False)

print('Linear Regression Coefficients')
print('(how much each unit increase adds to predicted cost)\n')
print(coef_df.to_string(index=False))

---
## 6. Model B — Random Forest

**What it does:** Builds 100 decision trees and averages their predictions.  
**Strength:** Captures complex, non-linear patterns — far more powerful than linear regression.  
**Limitation:** Less interpretable — you can't read a simple formula from it.

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

mae_rf  = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf   = r2_score(y_test, y_pred_rf)

print('RANDOM FOREST RESULTS')
print('-' * 35)
print(f'R²   (% variation explained): {r2_rf*100:.1f}%')
print(f'MAE  (average $ error):       ${mae_rf:,.0f}')
print(f'RMSE (error penalising outliers): ${rmse_rf:,.0f}')

---
## 7. Feature importance — what actually drives claim costs?

In [ ]:
importances = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#E74C3C' if i == importances.index[-1] else '#4472C4'
          for i in importances.index]
importances.plot(kind='barh', color=colors, ax=ax)
ax.set_title('Feature importance — what drives claim costs?',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance score')
plt.tight_layout()
plt.savefig('chart4_feature_importance.png', dpi=150)
plt.show()

print('Top cost driver:', importances.index[-1])
print(f'Importance score: {importances.iloc[-1]:.3f}')
print(f'\nThis means "{importances.index[-1]}" explains {importances.iloc[-1]*100:.0f}% of cost variation')

---
## 8. Model comparison — Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, preds, title, color, r2 in zip(
    axes,
    [y_pred_lr, y_pred_rf],
    ['Linear Regression', 'Random Forest'],
    ['#4472C4', '#2ECC71'],
    [r2_lr, r2_rf]
):
    ax.scatter(y_test, preds, alpha=0.4, color=color, s=20)
    ax.plot([y_test.min(), y_test.max()],
            [y_test.min(), y_test.max()],
            'r--', linewidth=1.5, label='Perfect prediction')
    ax.set_xlabel('Actual claim cost (USD)')
    ax.set_ylabel('Predicted claim cost (USD)')
    ax.set_title(f'{title}\nR² = {r2*100:.1f}%', fontweight='bold')
    ax.legend()

plt.suptitle('Actual vs Predicted — Model Comparison', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('chart5_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

print('How to read this chart:')
print('Dots close to the red dashed line = accurate predictions')
print('Dots far from the line = prediction errors')
print('Random Forest hugs the line more tightly = better model')

---
## 9. Model comparison summary

In [ ]:
summary = pd.DataFrame({
    'Metric': ['R² (% variance explained)', 'MAE (avg $ error)', 'RMSE'],
    'Linear Regression': [f'{r2_lr*100:.1f}%', f'${mae_lr:,.0f}', f'${rmse_lr:,.0f}'],
    'Random Forest':     [f'{r2_rf*100:.1f}%', f'${mae_rf:,.0f}', f'${rmse_rf:,.0f}']
})

summary.style.set_properties(**{'text-align': 'center'}).hide(axis='index')

---
## 10. Predict cost for a new patient

In [ ]:
# Change these values to predict cost for any patient profile
new_patient = pd.DataFrame({
    'age':      [45],
    'sex':      [1],       # 0=female, 1=male
    'bmi':      [28.5],
    'children': [2],
    'smoker':   [0],       # 0=no, 1=yes
    'region':   [2]        # 0-3 encoded region
})

predicted = rf.predict(new_patient)[0]

print('Patient profile:')
print('  Age: 45 | Sex: Male | BMI: 28.5 | Children: 2 | Non-smoker')
print(f'\nPredicted annual claim cost: ${predicted:,.0f}')
print('\nTry changing smoker from 0 to 1 and re-run — see how much the cost jumps.')

---
## 11. Key findings & implications for Nigerian health finance

### What the data shows

1. **Smoking is the dominant cost driver** — importance score 0.60, generating claims ~3.8x higher than non-smokers. This holds consistently across all age groups.

2. **BMI is the second predictor** (0.21) — relevant for chronic disease risk (diabetes, hypertension), which are high-burden conditions in Nigeria.

3. **Age escalates costs consistently** — the 55+ group costs ~2.1x the under-25 group, justifying age-banded premium structures.

4. **Sex and region have minimal impact** — lifestyle factors overwhelmingly outweigh demographics.

5. **Random Forest (R² = 88.3%) substantially outperforms Linear Regression (80.7%)** — complex interactions between risk factors exist that a linear model cannot capture.

### Implications for NHIA / Nigerian HMOs

- Risk-adjusted premiums should weight smoking status and BMI heavily
- Wellness intervention programs targeting smokers would yield the highest cost-containment ROI
- Age-banded pricing is actuarially justified by the data
- When applied to Nigerian NHIS data, replace US region with LGA/State, add ICD-10 diagnosis codes and facility type for a locally actionable model

---
*Built by Hillary Onah | Finance & Data Science Analyst, DHIN | June 2026*